# STEP-Risk vs O'Brien (STS) — AUROC comparison & statistical testing

Reproduces the updated analysis:

1. **Master AUROC table** with 95% bootstrap CI — each model scored on the patients it can serve.
2. **Pairwise difference tests** — *primary*: unpaired bootstrap, each model on its full applicable population, two-sided; *sensitivity*: paired bootstrap on the common STS-eligible subset.
3. **Forest plots** of ΔAUROC (STEP-Risk − O'Brien) with 95% CI, post-surgery and pre-op.

**Method decisions**
- **Full applicable population (primary).** STEP-Risk applies to the whole cohort; O'Brien is defined only for the 3 major cardiac procedures it was developed on. We compare each model on its own applicable population (unpaired bootstrap of the AUROC difference). A paired bootstrap on the common subset is reported as a sensitivity check.
- **Two-sided test + two-sided 95% CI (primary)** so that "significant" and "CI excludes 0" always agree; the one-sided p (directional hypothesis STEP-Risk > O'Brien) is retained as a supplementary column.

Models: **post** = STEP-Risk post-surgery, **pre** = STEP-Risk pre-operative, **sts** = O'Brien/STS score.
Seed = 42, 10,000 bootstrap replicates throughout.

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from statsmodels.stats.multitest import multipletests

# ---- paths ----
BASE = 'Risk_scores_and_true_outcome'   # folder with the risk-score + outcome CSVs
OUT  = 'analysis_outputs'               # where tables/figures are written
os.makedirs(OUT, exist_ok=True)

N_BOOT = 10000
SEED   = 42

AEnames  = ['mortality','stroke','renal','cpvntlng','reop','major','PLOS','SLOS']
STS_COLS = ['predmort','predstro','predrenf','predvent','predreop','predmm','pred14d','pred6d']
COHORTS  = ['retrospective_MMC','prospective_MMC','external_BWH']

COH_LABEL = {'retrospective_MMC':'MMC (retro)',
             'prospective_MMC':'MMC (temporal)',
             'external_BWH':'External (BWH)'}

MODEL_LABEL = {'post':'STEP-Risk (post-surgery)','pre':'STEP-Risk (pre-op)','sts':"O'Brien (STS)"}

OUT_LABEL = {'mortality':'Mortality','stroke':'Stroke','renal':'Renal failure',
             'cpvntlng':'Prolonged ventilation','reop':'Reoperation','major':'Major morbidity',
             'PLOS':'Prolonged LOS','SLOS':'Short LOS'}

## 1. Load the three cohorts

Each cohort has: true outcomes, STEP-Risk post-surgery probs, STEP-Risk pre-op probs, and O'Brien/STS probs. The STS columns are renamed to the 8 outcome names in a fixed order.

In [2]:
def load_cohort(which):
    if which == 'external_BWH':
        outcomes = pd.read_csv(f'{BASE}/BWH_outcomes.csv', index_col=0).iloc[:, 1:]
        RF  = pd.read_csv(f'{BASE}/Result_probs_topn_features_postsurg_BWH_pros.csv', index_col=0)
        pre = pd.read_csv(f'{BASE}/Result_probs_preop_features_BWH_pros.csv', index_col=0)
        sts = pd.read_csv(f'{BASE}/BWH_stspred.csv', index_col=0)
    elif which == 'retrospective_MMC':
        outcomes = pd.read_csv(f'{BASE}/internal_Ytest.csv', index_col=0).reset_index(drop=True)
        RF  = pd.read_csv(f'{BASE}/Result_probs_topn_features_postsurg.csv', index_col=0)
        pre = pd.read_csv(f'{BASE}/Result_probs_preop_features.csv', index_col=0)
        sts = pd.read_csv(f'{BASE}/retrospective_MMC_stspred_test.csv'); sts.columns = sts.columns.str.lower()
    elif which == 'prospective_MMC':
        outcomes = pd.read_csv(f'{BASE}/pros_outcomes.csv', index_col=0).reset_index(drop=True)
        RF  = pd.read_csv(f'{BASE}/Result_probs_topn_features_postsurg_pros.csv', index_col=0)
        pre = pd.read_csv(f'{BASE}/Result_probs_preop_features_pros.csv', index_col=0)
        sts = pd.read_csv(f'{BASE}/prospective_MMC_stspred.csv'); sts.columns = sts.columns.str.lower()
    else:
        raise ValueError(which)
    sts = sts[STS_COLS].copy(); sts.columns = AEnames
    return (outcomes.reset_index(drop=True), RF.reset_index(drop=True),
            pre.reset_index(drop=True), sts.reset_index(drop=True))

DATA = {w: load_cohort(w) for w in COHORTS}

# quick sanity: N, event counts, STS coverage
for w in COHORTS:
    o, RF, pre, sts = DATA[w]
    print(f'{w:20s} N={len(o):6d}  STS-available={int(sts[AEnames[0]].notna().sum()):6d}')

retrospective_MMC    N=  1969  STS-available=  1290
prospective_MMC      N=  2305  STS-available=  1787
external_BWH         N= 10170  STS-available=  6745


## 2. AUC helpers

`fast_auc` is a vectorised Mann–Whitney AUC (tie-corrected) — identical to `sklearn.roc_auc_score` but ~20× faster inside a 10,000-iteration bootstrap.

In [3]:
def fast_auc(y_true, y_score):
    y_true = np.asarray(y_true)
    n1 = y_true.sum(); n0 = len(y_true) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(y_score)
    return (r[y_true == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

# equivalence check vs sklearn
_o, _RF, _pre, _sts = DATA['external_BWH']
for AE in ['mortality','major']:
    y = _o[AE].values.astype(float); s = _RF[AE].values.astype(float)
    assert abs(fast_auc(y, s) - roc_auc_score(y, s)) < 1e-9
print('fast_auc matches sklearn roc_auc_score')

fast_auc matches sklearn roc_auc_score


## 3. Bootstrap functions

- `auc_ci` — percentile 95% CI for a single model's AUROC, resampling its own non-NaN patients.
- `unpaired_boot_compare` (**primary**) — resamples each model **independently on its own applicable population**; the AUROC difference CI/p come from the bootstrap distribution of (AUC_a − AUC_b).
- `paired_boot_compare` (**sensitivity**) — restricts to patients scored by **both** models and, in each replicate, scores the *same* resampled patients with both models.

p-values: `p_one` = one-sided (H1: model_a > model_b) = mean(diff ≤ 0); `p_two` = 2·min(left tail, right tail), capped at 1.

In [4]:
def auc_ci(y_true, y_prob, n_boot=N_BOOT, seed=SEED):
    y_true = np.asarray(y_true, float); y_prob = np.asarray(y_prob, float)
    m = ~(np.isnan(y_true) | np.isnan(y_prob))
    y_true, y_prob = y_true[m], y_prob[m]
    n = len(y_true); npos = int(y_true.sum())
    auc = fast_auc(y_true, y_prob)
    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        boots[b] = fast_auc(y_true[idx], y_prob[idx])
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    return dict(auc=auc, lo=lo, hi=hi, n=n, npos=npos)

In [5]:
def unpaired_boot_compare(y_true, prob_a, prob_b, n_boot=N_BOOT, seed=SEED):
    y = np.asarray(y_true, float); a = np.asarray(prob_a, float); b = np.asarray(prob_b, float)
    ma = ~(np.isnan(y) | np.isnan(a)); mb = ~(np.isnan(y) | np.isnan(b))
    ya, aa = y[ma], a[ma]; yb, bb = y[mb], b[mb]
    auc_a, auc_b = fast_auc(ya, aa), fast_auc(yb, bb)
    na, nb = len(ya), len(yb)
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for k in range(n_boot):
        ia = rng.integers(0, na, na); ib = rng.integers(0, nb, nb)
        diffs[k] = fast_auc(ya[ia], aa[ia]) - fast_auc(yb[ib], bb[ib])
    lo, hi   = np.nanpercentile(diffs, [2.5, 97.5])
    lo90, _  = np.nanpercentile(diffs, [5, 95])
    p_one = float(np.mean(diffs <= 0))
    p_two = float(min(1.0, 2*min(np.mean(diffs <= 0), np.mean(diffs >= 0))))
    return dict(auc_a=auc_a, auc_b=auc_b, n_a=na, n_b=nb,
                npos_a=int(ya.sum()), npos_b=int(yb.sum()),
                mean_diff=auc_a-auc_b, ci_lo=lo, ci_hi=hi, ci_lo90=lo90,
                p_one=p_one, p_two=p_two)

In [6]:
def paired_boot_compare(y_true, prob_a, prob_b, n_boot=N_BOOT, seed=SEED):
    y = np.asarray(y_true, float); a = np.asarray(prob_a, float); b = np.asarray(prob_b, float)
    m = ~(np.isnan(y) | np.isnan(a) | np.isnan(b))
    y, a, b = y[m], a[m], b[m]
    n = len(y); npos = int(y.sum())
    auc_a, auc_b = fast_auc(y, a), fast_auc(y, b)
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for k in range(n_boot):
        idx = rng.integers(0, n, n)
        yy = y[idx]
        diffs[k] = fast_auc(yy, a[idx]) - fast_auc(yy, b[idx])
    lo, hi = np.nanpercentile(diffs, [2.5, 97.5])
    p_one = float(np.mean(diffs <= 0))
    p_two = float(min(1.0, 2*min(np.mean(diffs <= 0), np.mean(diffs >= 0))))
    return dict(auc_a=auc_a, auc_b=auc_b, n=n, npos=npos, mean_diff=auc_a-auc_b,
                ci_lo=lo, ci_hi=hi, p_one=p_one, p_two=p_two)

## 4. Master AUROC table (per cohort × outcome × model)

Each model's AUROC with a 95% bootstrap CI, computed on that model's own non-NaN patients. `n` / `npos` show the applicable-population size and event count.

In [7]:
MODEL_SRC = {'post': 1, 'pre': 2, 'sts': 3}   # index into DATA tuple (o, RF, pre, sts)

rows = []
for w in COHORTS:
    tup = DATA[w]; o = tup[0]
    for AE in AEnames:
        y = o[AE].values.astype(float)
        for mkey, si in MODEL_SRC.items():
            prob = tup[si][AE].values.astype(float)
            r = auc_ci(y, prob)
            rows.append(dict(cohort=w, outcome=AE, model=mkey,
                             auc=r['auc'], lo=r['lo'], hi=r['hi'], n=r['n'], npos=r['npos']))
auc_df = pd.DataFrame(rows)
auc_df['AUROC (95% CI)'] = auc_df.apply(lambda r: f"{r.auc:.3f} ({r.lo:.3f}–{r.hi:.3f})", axis=1)

auc_df.to_csv(f'{OUT}/auroc_master_table.csv', index=False)

# wide view: rows = cohort × outcome, cols = the three models
wide = (auc_df.assign(model=auc_df.model.map({'pre':'STEP-Risk preop','post':'STEP-Risk postsurg','sts':"O'Brien (STS)"}))
        .pivot_table(index=['cohort','outcome'], columns='model', values='AUROC (95% CI)', aggfunc='first'))
wide = wide[['STEP-Risk preop','STEP-Risk postsurg',"O'Brien (STS)"]]
wide.to_csv(f'{OUT}/auroc_table_wide.csv')
wide

model                            STEP-Risk preop   STEP-Risk postsurg  \
cohort            outcome                                               
external_BWH      PLOS       0.910 (0.896–0.923)  0.898 (0.883–0.912)   
                  SLOS       0.777 (0.765–0.790)  0.759 (0.746–0.771)   
                  cpvntlng   0.850 (0.838–0.863)  0.881 (0.871–0.892)   
                  major      0.780 (0.767–0.793)  0.825 (0.813–0.837)   
                  mortality  0.869 (0.847–0.889)  0.893 (0.876–0.908)   
                  renal      0.844 (0.823–0.863)  0.892 (0.878–0.906)   
                  reop       0.726 (0.697–0.753)  0.746 (0.720–0.772)   
                  stroke     0.705 (0.676–0.734)  0.722 (0.694–0.750)   
prospective_MMC   PLOS       0.854 (0.824–0.882)  0.869 (0.843–0.894)   
                  SLOS       0.799 (0.781–0.817)  0.817 (0.800–0.834)   
                  cpvntlng   0.904 (0.880–0.926)  0.904 (0.880–0.928)   
                  major      0.808 (0.777–0.837)  0.834 (0.805–0.861)   
                  mortality  0.873 (0.829–0.911)  0.880 (0.835–0.918)   
                  renal      0.889 (0.841–0.929)  0.891 (0.839–0.933)   
                  reop       0.802 (0.760–0.842)  0.815 (0.775–0.852)   
                  stroke     0.773 (0.696–0.845)  0.792 (0.716–0.862)   
retrospective_MMC PLOS       0.853 (0.826–0.879)  0.878 (0.852–0.902)   
                  SLOS       0.769 (0.748–0.790)  0.782 (0.762–0.802)   
                  cpvntlng   0.880 (0.852–0.906)  0.896 (0.871–0.919)   
                  major      0.806 (0.776–0.835)  0.827 (0.798–0.855)   
                  mortality  0.860 (0.813–0.903)  0.873 (0.825–0.915)   
                  renal      0.834 (0.770–0.889)  0.879 (0.828–0.923)   
                  reop       0.704 (0.657–0.750)  0.726 (0.676–0.774)   
                  stroke     0.755 (0.674–0.830)  0.770 (0.688–0.846)   

model                              O'Brien (STS)  
cohort            outcome                         
external_BWH      PLOS       0.834 (0.787–0.876)  
                  SLOS       0.682 (0.663–0.701)  
                  cpvntlng   0.814 (0.789–0.837)  
                  major      0.750 (0.728–0.771)  
                  mortality  0.841 (0.799–0.880)  
                  renal      0.807 (0.768–0.843)  
                  reop       0.597 (0.553–0.640)  
                  stroke     0.712 (0.667–0.755)  
prospective_MMC   PLOS       0.851 (0.814–0.885)  
                  SLOS       0.798 (0.778–0.819)  
                  cpvntlng   0.821 (0.779–0.863)  
                  major      0.773 (0.733–0.812)  
                  mortality  0.764 (0.676–0.845)  
                  renal      0.849 (0.771–0.912)  
                  reop       0.657 (0.595–0.716)  
                  stroke     0.700 (0.575–0.811)  
retrospective_MMC PLOS       0.893 (0.856–0.925)  
                  SLOS       0.762 (0.736–0.788)  
                  cpvntlng   0.846 (0.804–0.885)  
                  major      0.763 (0.720–0.804)  
                  mortality  0.790 (0.710–0.862)  
                  renal      0.807 (0.694–0.908)  
                  reop       0.649 (0.577–0.719)  
                  stroke     0.743 (0.638–0.839)

In [28]:
cohort_order = [
    "retrospective_MMC",
    "prospective_MMC",
    "external_BWH",
]

outcome_order = [
    "mortality",
    "stroke",
    "renal",
    "cpvntlng",
    "reop",
    "major",
    "PLOS",
    "SLOS",
]

wide = (
    wide.reset_index()
      .assign(
          cohort=pd.Categorical(
              wide.reset_index()["cohort"],
              categories=cohort_order,
              ordered=True,
          ),
          outcome=pd.Categorical(
              wide.reset_index()["outcome"],
              categories=outcome_order,
              ordered=True,
          ),
      )
      .sort_values(["cohort", "outcome"])
      .set_index(["cohort", "outcome"])
)
wide = wide[['O\'Brien (STS)','STEP-Risk preop', 'STEP-Risk postsurg']]
wide.to_csv(f'{OUT}/auroc_table_wide.csv')


## 5. Pairwise comparisons

Three pairs — (post vs sts), (pre vs sts), (post vs pre) — under both schemes. The primary read is **unpaired**; **paired** is the sensitivity check.

In [10]:
PAIRS = [('post','sts'), ('pre','sts'), ('post','pre')]

unp_rows, pair_rows = [], []
for w in COHORTS:
    tup = DATA[w]; o = tup[0]
    for AE in AEnames:
        y = o[AE].values.astype(float)
        for ma, mb in PAIRS:
            pa = tup[MODEL_SRC[ma]][AE].values.astype(float)
            pb = tup[MODEL_SRC[mb]][AE].values.astype(float)
            u = unpaired_boot_compare(y, pa, pb); u.update(cohort=w, outcome=AE, model_a=ma, model_b=mb)
            unp_rows.append(u)
            p = paired_boot_compare(y, pa, pb);   p.update(cohort=w, outcome=AE, model_a=ma, model_b=mb)
            pair_rows.append(p)
unp_df = pd.DataFrame(unp_rows)
cmp_df = pd.DataFrame(pair_rows)
print('unpaired rows:', len(unp_df), ' paired rows:', len(cmp_df))

unpaired rows: 72  paired rows: 72


## 6. Final comparison tables (STEP-Risk vs O'Brien)

`build_final(model_a)` merges the primary (unpaired) and sensitivity (paired) results for one STEP-Risk variant vs O'Brien into a single reviewer-facing table, and adds a Benjamini–Hochberg FDR column over the 24 tests (3 cohorts × 8 outcomes).

In [ ]:
def stars(p):
    return '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''

def build_final(model_a):
    u = unp_df[(unp_df.model_a==model_a)&(unp_df.model_b=='sts')].copy()
    p = cmp_df[(cmp_df.model_a==model_a)&(cmp_df.model_b=='sts')].copy()
    p = p.rename(columns=dict(mean_diff='dAUROC_paired', ci_lo='cip_lo', ci_hi='cip_hi',
                              p_two='p_two_paired', n='N_paired'))
    m = u.merge(p[['cohort','outcome','dAUROC_paired','cip_lo','cip_hi','p_two_paired','N_paired']],
                on=['cohort','outcome'])
    m['q_two'] = np.nan
    for w in COHORTS:
        idx = m.cohort==w
        m.loc[idx,'q_two'] = multipletests(m.loc[idx,'p_two'], method='fdr_bh')[1]
    out = pd.DataFrame({
        'Cohort':  m.cohort.map(COH_LABEL),
        'Outcome': m.outcome.map(OUT_LABEL),
        'N_STEP':  m.n_a, 'N_OBrien': m.n_b,
        'AUROC_STEP':   m.auc_a.round(3),
        'AUROC_OBrien': m.auc_b.round(3),
        'dAUROC':  m.mean_diff.round(3),
        'CI95':    m.apply(lambda r: f"{r.ci_lo:.3f}–{r.ci_hi:.3f}", axis=1),
        'p_two':   m.p_two, 'q_two_BH': m.q_two.round(3), 'p_one': m.p_one,
        'sig':     m.p_two.map(stars),
        'dAUROC_paired': m.dAUROC_paired.round(3),
        'CI95_paired':   m.apply(lambda r: f"{r.cip_lo:.3f}–{r.cip_hi:.3f}", axis=1),
        'p_two_paired':  m.p_two_paired, 'N_paired': m.N_paired,
    })
    cohort_order = [COH_LABEL[c] for c in COHORTS]
    out['Cohort'] = pd.Categorical(out['Cohort'], cohort_order, ordered=True)
    out['Outcome'] = pd.Categorical(out['Outcome'], [OUT_LABEL[a] for a in AEnames], ordered=True)
    return out.sort_values(['Cohort','Outcome']).reset_index(drop=True)

final_post = build_final('post')
final_pre  = build_final('pre')
final_post.to_csv(f'{OUT}/diff_post_vs_obrien_FINAL.csv', index=False)
final_pre.to_csv(f'{OUT}/diff_pre_vs_obrien_FINAL.csv', index=False)
print('post-surgery: %d/24 two-sided sig' % (final_post.p_two<0.05).sum())
print('pre-op:       %d/24 two-sided sig' % (final_pre.p_two<0.05).sum())
final_post

## 7. Forest plots of ΔAUROC (STEP-Risk − O'Brien)

Primary (unpaired) ΔAUROC with two-sided 95% CI, one panel per cohort. Filled markers with a black edge = p < 0.05 (two-sided); faded = not significant. Positive = STEP-Risk discriminates better.

In [ ]:
def forest(model_a, fname, title):
    sub = unp_df[(unp_df.model_a==model_a)&(unp_df.model_b=='sts')].copy()
    order = AEnames[::-1]
    coh_order = ['retrospective_MMC','prospective_MMC','external_BWH']
    coh_short = {'retrospective_MMC':'Retrospective\n(temporal)','prospective_MMC':'Prospective','external_BWH':'External (BWH)'}
    colors = {'retrospective_MMC':'#4c72b0','prospective_MMC':'#dd8452','external_BWH':'#55a868'}

    fig, axes = plt.subplots(1, 3, figsize=(11,4.4), sharex=True, sharey=True)
    for j, w in enumerate(coh_order):
        ax = axes[j]; d = sub[sub.cohort==w].set_index('outcome').reindex(order)
        for i, AE in enumerate(order):
            r = d.loc[AE]; sig = r['p_two']<0.05; c = colors[w]
            ax.plot([r['ci_lo'],r['ci_hi']], [i,i], color=c, lw=2, alpha=0.9 if sig else 0.35, zorder=2)
            ax.scatter([r['mean_diff']], [i], s=45 if sig else 30, color=c,
                       edgecolor='black' if sig else 'none', linewidth=0.8, zorder=3, alpha=1 if sig else 0.5)
        ax.axvline(0, color='0.4', ls='--', lw=1, zorder=1)
        ax.set_yticks(np.arange(len(order)))
        ax.set_yticklabels([OUT_LABEL[a] for a in order] if j==0 else [])
        ax.set_title(coh_short[w], fontsize=9)
        ax.set_xlabel('ΔAUROC  (STEP-Risk − O\u2019Brien)', fontsize=8)
        ax.margins(y=0.05)
    fig.suptitle(title, fontsize=10.5, fontweight='bold', x=0.5, y=1.02)
    axes[2].text(0.97, 0.02, '\u2192 STEP-Risk better', transform=axes[2].transAxes,
                 ha='right', va='bottom', fontsize=7, color='0.3')
    leg = [Line2D([0],[0], marker='o', color='w', markerfacecolor='0.5', markeredgecolor='black', markersize=7, label='p < 0.05 (two-sided)'),
           Line2D([0],[0], marker='o', color='w', markerfacecolor='0.7', markeredgecolor='none', markersize=6, alpha=0.6, label='n.s.')]
    axes[0].legend(handles=leg, loc='lower left', fontsize=6.5, frameon=False)
    fig.tight_layout()
    fig.savefig(fname, dpi=300, bbox_inches='tight')
    return fig

_ = forest('post', f'{OUT}/forest_postsurg_vsobrien.png',
           'STEP-Risk (post-surgery) vs O\u2019Brien STS score  \u2014  each model on its full applicable population')

In [ ]:
_ = forest('pre', f'{OUT}/forest_preop_vsobrien.png',
           'STEP-Risk (pre-operative) vs O\u2019Brien STS score  \u2014  each model on its full applicable population')

## 8. Summary

- **Primary test** — unpaired bootstrap, two-sided, each model on its full applicable population.
- **Post-surgery STEP-Risk** beats O'Brien in **7/8** outcomes externally (all but stroke), **4/8** prospectively, **2/8** retrospectively (small event counts limit power).
- **Pre-op STEP-Risk** beats O'Brien in **5/8** externally and **3/8** prospectively; retrospectively no cell reaches significance but 7/8 point estimates favour STEP-Risk. Reoperation is the pre-op model's standout (O'Brien discriminates reoperation poorly).
- Unpaired and paired schemes agree on all 24 post-surgery cells; a few pre-op cells are borderline (noted in the report).

Outputs written to `analysis_outputs/`: `auroc_master_table.csv`, `auroc_table_wide.csv`, `diff_post_vs_obrien_FINAL.csv`, `diff_pre_vs_obrien_FINAL.csv`, `forest_postsurg_vsobrien.png`, `forest_preop_vsobrien.png`.